# Cross-Dataset Evaluation

This notebook evaluates models across datasets to measure domain shift robustness:
- Train on TorchSig (synthetic) → Evaluate on Panoradio (real HF)

Uses the modulation family abstraction for fair cross-dataset comparison.

In [ ]:
import sys
from pathlib import Path

src_path = Path("../src")
if src_path.exists():
    sys.path.insert(0, str(src_path.resolve()))

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import accuracy_score, confusion_matrix
from tqdm.auto import tqdm

from robust_amc.data import (
    get_loaders,
    load_config_from_yaml,
    FamilyMapper,
    load_panoradio_data,
    Compose,
    PowerNormalize,
)
from robust_amc.data.transforms import ToTensor, RandomCrop
from robust_amc.models import create_pfcnn, create_clsr_amc
from robust_amc.utils import get_device

## 1. Setup

In [ ]:
TORCHSIG_CONFIG = Path("../configs/datasets/torchsig_train.yaml")
PANORADIO_DIR = Path("../data/panoradio")
PANORADIO_MAP = Path("../configs/label_maps/panoradio_to_family.yaml")
CHECKPOINTS_DIR = Path("../checkpoints")

device = get_device("auto")
print(f"Using device: {device}")

In [ ]:
# Load TorchSig family names (what models were trained on)
ts_config = load_config_from_yaml(TORCHSIG_CONFIG)
ts_loaders = get_loaders(ts_config)
source_families = ts_loaders["family_names"]

print(f"Source (TorchSig) families: {source_families}")

## 2. Load Panoradio Target Dataset

In [ ]:
if PANORADIO_DIR.exists() and (PANORADIO_DIR / "rscd_2048.npy").exists():
    pano_data, pano_labels, pano_snrs = load_panoradio_data(PANORADIO_DIR)
    pano_mapper = FamilyMapper(PANORADIO_MAP)
    
    # Map to family indices
    pano_family_indices = np.array([pano_mapper.get_family_idx(str(lbl)) or -1 for lbl in pano_labels])
    valid_mask = pano_family_indices >= 0
    
    pano_data = pano_data[valid_mask]
    pano_family_indices = pano_family_indices[valid_mask]
    pano_snrs = pano_snrs[valid_mask]
    
    target_families = pano_mapper.family_names
    
    print(f"Panoradio Dataset:")
    print(f"  Samples: {len(pano_data)}")
    print(f"  Families: {target_families}")
    print(f"  SNR range: {pano_snrs.min():.0f} to {pano_snrs.max():.0f} dB")
    
    has_panoradio = True
else:
    print(f"Panoradio data not found at {PANORADIO_DIR}")
    print("Download from: https://panoradio-sdr.de/radio-signal-classification-dataset/")
    has_panoradio = False

## 3. Load Models

In [ ]:
models = {}

# Baseline
baseline_path = CHECKPOINTS_DIR / "pfcnn_torchsig" / "best_model.pt"
if baseline_path.exists():
    model = create_pfcnn(num_classes=len(source_families))
    ckpt = torch.load(baseline_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    models["Baseline"] = model
    print(f"Loaded: Baseline from {baseline_path}")

# MDA-DMC
mda_path = CHECKPOINTS_DIR / "pfcnn_augmented" / "best_model.pt"
if mda_path.exists():
    model = create_pfcnn(num_classes=len(source_families))
    ckpt = torch.load(mda_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    models["MDA-DMC"] = model
    print(f"Loaded: MDA-DMC from {mda_path}")

# CLSR-AMC
clsr_path = CHECKPOINTS_DIR / "clsr_amc" / "best_model.pt"
if clsr_path.exists():
    model = create_clsr_amc(num_classes=len(source_families))
    ckpt = torch.load(clsr_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    models["CLSR-AMC"] = model
    print(f"Loaded: CLSR-AMC from {clsr_path}")

if not models:
    print(f"No models found! Train on TorchSig first.")

## 4. Family Mapping Between Datasets

Map between TorchSig and Panoradio family indices.

In [ ]:
if has_panoradio:
    # Find common families
    common_families = [f for f in source_families if f in target_families]
    
    # Create mapping: source_idx -> target_idx for common families
    source_to_target = {}
    target_to_source = {}
    
    for family in common_families:
        src_idx = source_families.index(family)
        tgt_idx = target_families.index(family)
        source_to_target[src_idx] = tgt_idx
        target_to_source[tgt_idx] = src_idx
    
    print(f"Common families ({len(common_families)}): {common_families}")
    print(f"\nMapping (source -> target):")
    for src_idx, tgt_idx in source_to_target.items():
        print(f"  {source_families[src_idx]} ({src_idx}) -> {target_families[tgt_idx]} ({tgt_idx})")
    
    # Families only in source (will be OOD for target)
    source_only = [f for f in source_families if f not in target_families]
    if source_only:
        print(f"\nSource-only families (not in Panoradio): {source_only}")
    
    # Families only in target (model has never seen)
    target_only = [f for f in target_families if f not in source_families]
    if target_only:
        print(f"Target-only families (OOD for model): {target_only}")

## 5. Cross-Dataset Evaluation

In [ ]:
def evaluate_cross_dataset(model, data, labels, snrs, source_families, target_to_source, device, max_samples=5000):
    """Evaluate model on target dataset with family mapping."""
    model.eval()
    model = model.to(device)
    
    transform = Compose([
        RandomCrop(128),  # Crop to model input size
        PowerNormalize(),
        ToTensor(),
    ])
    
    predictions = []
    ground_truth = []
    snr_values = []
    
    # Subsample if needed
    indices = np.arange(len(data))
    if len(indices) > max_samples:
        indices = np.random.choice(indices, max_samples, replace=False)
    
    with torch.no_grad():
        for idx in tqdm(indices, desc="Evaluating"):
            target_label = labels[idx]
            
            # Skip if target family not in source
            if target_label not in target_to_source:
                continue
            
            source_label = target_to_source[target_label]
            
            # Prepare signal
            signal = data[idx]
            if np.iscomplexobj(signal):
                signal = np.stack([signal.real, signal.imag], axis=0).astype(np.float32)
            
            signal = transform(signal)
            x = signal.unsqueeze(0).to(device)
            
            # Predict
            outputs = model(x)
            pred = outputs.argmax(dim=1).item()
            
            predictions.append(pred)
            ground_truth.append(source_label)
            snr_values.append(snrs[idx])
    
    return np.array(predictions), np.array(ground_truth), np.array(snr_values)

In [ ]:
results = {}

if has_panoradio and models:
    for name, model in models.items():
        print(f"\nEvaluating: {name}")
        preds, gt, snrs_eval = evaluate_cross_dataset(
            model, pano_data, pano_family_indices, pano_snrs,
            source_families, target_to_source, device
        )
        
        accuracy = accuracy_score(gt, preds)
        results[name] = {
            "predictions": preds,
            "ground_truth": gt,
            "snrs": snrs_eval,
            "accuracy": accuracy,
        }
        print(f"  Overall accuracy: {accuracy:.2%}")

## 6. Accuracy vs SNR

In [ ]:
if results:
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#2563eb', '#dc2626', '#22c55e']
    
    for (name, result), color in zip(results.items(), colors):
        preds = result["predictions"]
        gt = result["ground_truth"]
        snrs = result["snrs"]
        
        # Compute accuracy per SNR bin
        snr_bins = sorted(set(int(s) for s in snrs))
        snr_accuracies = []
        
        for snr_val in snr_bins:
            mask = np.abs(snrs - snr_val) <= 2
            if mask.sum() > 10:
                acc = (preds[mask] == gt[mask]).mean()
                snr_accuracies.append((snr_val, acc))
        
        if snr_accuracies:
            x = [s[0] for s in snr_accuracies]
            y = [s[1] for s in snr_accuracies]
            ax.plot(x, y, '-o', linewidth=2, markersize=6, label=name, color=color)
    
    ax.set_xlabel("SNR (dB)")
    ax.set_ylabel("Accuracy")
    ax.set_title("Cross-Dataset: TorchSig → Panoradio")
    ax.set_ylim(0, 1)
    ax.axhline(y=1/len(common_families), color='gray', linestyle=':', alpha=0.5, 
               label=f'Random ({len(common_families)} classes)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. Confusion Matrix

In [ ]:
if results:
    n_models = len(results)
    fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
    if n_models == 1:
        axes = [axes]
    
    for ax, (name, result) in zip(axes, results.items()):
        cm = confusion_matrix(result["ground_truth"], result["predictions"], 
                             labels=range(len(source_families)))
        
        # Only show common families (non-zero rows/cols)
        common_idx = sorted(target_to_source.values())
        cm_common = cm[np.ix_(common_idx, common_idx)]
        cm_norm = cm_common.astype('float') / (cm_common.sum(axis=1, keepdims=True) + 1e-8)
        
        im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
        
        common_names = [source_families[i] for i in common_idx]
        ax.set_xticks(range(len(common_names)))
        ax.set_yticks(range(len(common_names)))
        ax.set_xticklabels(common_names, rotation=45, ha='right', fontsize=9)
        ax.set_yticklabels(common_names, fontsize=9)
        
        for i in range(len(common_names)):
            for j in range(len(common_names)):
                value = cm_norm[i, j]
                color = "white" if value > 0.5 else "black"
                ax.text(j, i, f"{value:.2f}", ha="center", va="center", 
                       color=color, fontsize=8)
        
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(f"{name}: {result['accuracy']:.1%}")
    
    plt.suptitle("Cross-Dataset Confusion Matrices", fontsize=12)
    plt.tight_layout()
    plt.show()

## 8. Domain Gap Analysis

In [ ]:
# Compare in-domain (TorchSig test) vs cross-domain (Panoradio) accuracy
if results and models:
    print("\nDomain Gap Analysis:")
    print("=" * 60)
    print(f"{'Model':<15} {'TorchSig':>12} {'Panoradio':>12} {'Gap':>10}")
    print("-" * 60)
    
    # Evaluate on TorchSig test set
    for name, model in models.items():
        model.eval()
        model = model.to(device)
        
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch in ts_loaders["test"]:
                x, y, _ = batch
                x = x.to(device)
                y = y.to(device)
                
                outputs = model(x)
                preds = outputs.argmax(dim=1)
                correct += (preds == y).sum().item()
                total += len(y)
        
        ts_acc = correct / total
        pano_acc = results[name]["accuracy"]
        gap = ts_acc - pano_acc
        
        print(f"{name:<15} {ts_acc:>12.2%} {pano_acc:>12.2%} {gap:>10.2%}")
    
    print("\nInterpretation:")
    print("  - Gap measures accuracy drop from synthetic to real data")
    print("  - Smaller gap indicates better domain generalization")
    print("  - MDA-DMC and contrastive learning typically reduce the gap")

## 9. Summary

In [ ]:
if results:
    print("\nCross-Dataset Evaluation Summary")
    print("=" * 50)
    print(f"Source: TorchSig (synthetic)")
    print(f"Target: Panoradio (real HF)")
    print(f"Common families: {common_families}")
    print("\nResults:")
    print("-" * 30)
    
    for name, result in sorted(results.items(), key=lambda x: -x[1]['accuracy']):
        print(f"  {name:<15} {result['accuracy']:>8.2%}")
    
    best = max(results.items(), key=lambda x: x[1]['accuracy'])
    print(f"\nBest cross-domain model: {best[0]} ({best[1]['accuracy']:.2%})")
else:
    print("No results to summarize. Ensure Panoradio data is available and models are trained.")